# mv-PIKN · Wilson–Cowan-inspired regularizer
**Explicit formulation, numerical checks, and a training-only smoke test**

Run **Sections 1–8 in order**. CPU is sufficient; GPU is optional. No previous notebook has to be open. This notebook embeds the checked loader and core model. It uses the existing processed run, freshly initializes the model, and checks three updates on at most six training examples. No validation or outer-test samples are used.

The component is a **phenomenological dynamics prior**, not a physiological measurement or a verified model of seizure generation. It uses a reduced sigmoid-rate equation inspired by excitatory/inhibitory interactions. The signs are constrained by construction; the coordinates and effective time constants are not biologically identified. This is a proposed, fully documented architecture amendment, not an exact reproduction of the manuscript.

The original Wilson–Cowan formulation motivates interacting population dynamics; our fixed state mapping, slow effective time constants, shared parameters, reduced equations and loss coupling are engineering choices. [Original paper](https://pubmed.ncbi.nlm.nih.gov/4332108/); [full text](https://pmc.ncbi.nlm.nih.gov/articles/PMC1484078/).

## 1 · The equations and what they mean
For the current fused latent vector $z_t\in\mathbb{R}^{32}$, define the **fixed**, elementwise map

$$s_t=\sigma(z_t)=(E_t,I_t),\qquad E_t,I_t\in(0,1)^{16}.$$

No extra trainable projection can hide disagreement between the two dynamics models. The first and second halves are formal E/I coordinates with imposed interaction signs, not measured E/I populations. Each of the 16 pairs follows the same two equations; there is no anatomical connectivity claim:

$$\tau_E\dot E=-E+\sigma(w_{EE}E-w_{EI}I+b_E),$$
$$\tau_I\dot I=-I+\sigma(w_{IE}E-w_{II}I+b_I).$$

The four $w$ values are nonnegative magnitudes; inhibitory terms carry explicit minus signs. All pairs share **eight scalar parameters**: four weights, two time constants, two net drives. Net drives combine drive/threshold effects rather than trying to identify them separately.

Parameterizations: $w=4\sigma(a)$, $\tau=4+36\sigma(c)$ effective seconds, and $b=2\tanh(d)$. Initial weights are [1.5, 1, 1, 0.5], time constants [10,16] seconds, drives [-0.5,-0.5]. Bounds and initialization are **untuned prototype choices**. These slow constants describe four-second EEG-window features; they must not be described as neuronal membrane time constants.

This reduced rate model omits the original refractory terms and offset response conventions. Its biological meaning requires separate validation. Shared parameters also restrict expressivity, which should be assessed by ablation.

## 2 · Dependencies and existing data

In [ ]:
%pip -q install numpy pandas h5py
import importlib.util
if importlib.util.find_spec('torch') is None:
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','install','torch'])
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
"""Fold-scoped, causal CHB-MIT loading. No model fitting or alarm scoring."""
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import h5py

class FoldData:
    def __init__(self, run, outer='outer_chb01', scope='inner_01'):
        self.run = Path(run)
        self.cfg = json.loads((self.run/'config.json').read_text())
        self.inventory = json.loads((self.run/'locked_inventory.json').read_text())
        self.events = json.loads((self.run/'locked_events.json').read_text())
        folds = json.loads((self.run/'locked_folds.json').read_text())
        self.fold = next(f for f in folds if f['id'] == outer)
        self.outer, self.scope = outer, scope
        if scope == 'outer_refit':
            self.roles = {'train': self.fold['development_subjects'], 'test': self.fold['test_subjects']}
        else:
            inner = next(f for f in self.fold['inner_folds'] if f['id'] == scope)
            self.roles = {'train': inner['train_subjects'], 'validation': inner['validation_subjects']}
        if set(self.roles['train']) & set(self.fold['test_subjects']):
            raise ValueError('Held-out subject appears in training.')
        if len(set(sum(self.roles.values(), []))) != len(sum(self.roles.values(), [])):
            raise ValueError('Roles overlap or contain duplicate subjects.')
        status = json.loads((self.run/'status.json').read_text())
        if status.get('status') not in ['review_required','preprocessing_complete']:
            raise ValueError('Preprocessing is incomplete.')
        blocking = [x for x in status.get('cohort_issues', [])
                    if x.get('reason') != 'no_valid_preictal_input_after_QC']
        if blocking:
            raise ValueError(f'Unresolved preprocessing issues: {blocking}')
        if status.get('cohort_issues'):
            audit = json.loads((self.run/'training_label_audit_status_v2_2.json').read_text())
            if audit.get('revision') != '2.2':
                raise ValueError('Run preprocessing Section 7B first.')
        self.scaler_path = self.run/'folds'/outer/scope/'train_scaler.npz'
        with np.load(self.scaler_path, allow_pickle=False) as z:
            if set(z['training_subjects'].tolist()) != set(self.roles['train']):
                raise ValueError('Scaler training subjects do not match the fold.')
            if set(z['test_subjects'].tolist()) != set(self.fold['test_subjects']):
                raise ValueError('Scaler outer test subjects do not match.')
            self.scaler = {k:z[k].copy() for k in ['raw_mean','raw_std','feature_mean','feature_std']}
        C = len(self.cfg['channels'])
        for k,shape in [('raw_mean',(C,)),('raw_std',(C,)),('feature_mean',(C,5)),('feature_std',(C,5))]:
            v = self.scaler[k]
            if v.shape != shape or not np.isfinite(v).all() or ('std' in k and np.any(v<=0)):
                raise ValueError(f'Invalid scaler: {k}')
        ratio = self.cfg['fusion_history_s']/self.cfg['window_s']
        if ratio != int(ratio):
            raise ValueError('Fusion history must contain whole windows.')
        self.history = int(ratio)
        self.subject_index = {s:i for i,s in enumerate(sorted({r['subject'] for r in self.inventory}))}
        self._metadata = {}

    def records(self, role):
        if role not in self.roles:
            raise ValueError(f'{role} is unavailable in {self.scope}; allowed: {list(self.roles)}')
        rows = [r for r in self.inventory if r['subject'] in self.roles[role]
                and r['signal_ok'] and not r['excluded_case'] and r['split']=='pool']
        return sorted(rows, key=lambda r:(r['subject'],r['case'],r['start_s'],r['relative_path']))

    def path(self, row):
        return self.run/'processed'/row['case']/'pool'/(Path(row['relative_path']).stem+'.h5')

    def metadata(self, row):
        p = self.path(row)
        if p not in self._metadata:
            with h5py.File(p,'r') as h:
                if str(h.attrs['case']) != row['case'] or json.loads(h.attrs['channels_json']) != self.cfg['channels']:
                    raise ValueError(f'Unexpected case or channels: {p}')
                if json.loads(h.attrs['config_json']) != self.cfg:
                    raise ValueError(f'Unexpected processing configuration: {p}')
                keys = ['label','valid_signal','window_start_s','decision_s','pair_with_previous']
                m = {k:h[k][:] for k in keys}
            n = len(m['label']); w = self.cfg['window_s']
            if any(len(v)!=n for v in m.values()) or not np.isin(m['label'],[-1,0,1,2]).all():
                raise ValueError(f'Invalid metadata: {p}')
            if n and (not np.allclose(m['decision_s']-m['window_start_s'],w) or
                      np.any(np.diff(m['decision_s'])<=0)):
                raise ValueError(f'Invalid timeline: {p}')
            # History ends before current window starts. Reset on every EDF boundary,
            # missing interval, or invalid signal; never concatenate across files.
            ready = np.zeros(n,dtype=bool)
            streak = 0
            for i in range(n):
                if i and not np.isclose(m['window_start_s'][i],m['decision_s'][i-1],rtol=0,atol=1e-6):
                    streak = 0
                if m['valid_signal'][i]:
                    ready[i] = streak >= self.history
                    streak += 1
                else:
                    streak = 0
            m['prediction_ready'] = ready
            self._metadata[p] = m
        return self._metadata[p]

    def index(self, role, mode='supervised'):
        if mode not in ['supervised','pairs','replay']:
            raise ValueError(mode)
        if mode=='pairs' and role!='train':
            raise ValueError('Future-window pairs are restricted to training.')
        items=[]
        for row in self.records(role):
            m=self.metadata(row); n=len(m['label'])
            if mode=='replay': keep=np.ones(n,dtype=bool)
            else:
                keep=m['prediction_ready'] & (m['label']>=0)
                if mode=='pairs':
                    next_ok=np.zeros(n,dtype=bool)
                    if n>1:
                        next_ok[:-1]=m['valid_signal'][1:] & m['pair_with_previous'][1:]
                    keep &= next_ok
            items.extend((row,int(i)) for i in np.flatnonzero(keep))
        return items

    def read(self, row, i, mode='supervised'):
        if mode not in ['supervised','pairs','replay']: raise ValueError(mode)
        m=self.metadata(row); ready=bool(m['prediction_ready'][i])
        if mode!='replay' and (not ready or m['label'][i]<0):
            raise ValueError('Supervised sample lacks labels or causal history.')
        C=len(self.cfg['channels']); L=int(self.cfg['fs']*self.cfg['window_s']); H=self.history
        inputs={'raw_t':np.zeros((C,L),np.float32), 'spec_t':np.zeros((C,5),np.float32),
                'raw_past':np.zeros((H,C,L),np.float32), 'spec_past':np.zeros((H,C,5),np.float32)}
        def normal(raw,spec):
            s=self.scaler
            x=((raw-s['raw_mean'][:,None])/s['raw_std'][:,None]).astype(np.float32)
            f=((spec-s['feature_mean'])/s['feature_std']).astype(np.float32)
            if not np.isfinite(x).all() or not np.isfinite(f).all():
                raise ValueError('Nonfinite normalized input')
            return x,f
        auxiliary={}
        with h5py.File(self.path(row),'r') as h:
            # Unavailable replay rows retain timestamps but carry zero placeholders.
            # The model must run ONLY when prediction_ready is true.
            if ready:
                inputs['raw_t'],inputs['spec_t']=normal(h['raw_uv'][i],h['features'][i])
                inputs['raw_past'],inputs['spec_past']=normal(h['raw_uv'][i-H:i],h['features'][i-H:i])
            if mode=='pairs':
                if row['subject'] not in self.roles['train']:
                    raise ValueError('Future-window pairs are restricted to training subjects.')
                if i+1>=len(m['label']) or not m['valid_signal'][i+1] or not m['pair_with_previous'][i+1]:
                    raise ValueError('Invalid adjacent pair.')
                auxiliary['raw_tdt'],auxiliary['spec_tdt']=normal(h['raw_uv'][i+1],h['features'][i+1])
        return dict(inputs=inputs, auxiliary=auxiliary,
                    target=dict(label=int(m['label'][i])),
                    metadata=dict(subject=row['subject'],case=row['case'],patient_idx=self.subject_index[row['subject']],
                                  file=str(self.path(row)),index=i,start_s=float(m['window_start_s'][i]),
                                  decision_s=float(m['decision_s'][i]),valid_signal=bool(m['valid_signal'][i]),
                                  prediction_ready=ready, reset_at_file_start=i==0))

    def iterate(self, role, mode='replay', shuffle=False, seed=20261028):
        if shuffle and (role!='train' or mode=='replay'):
            raise ValueError('Only supervised training may be shuffled.')
        items=self.index(role,mode)
        if shuffle:
            rng=np.random.default_rng(seed); rng.shuffle(items)
        for row,i in items: yield self.read(row,i,mode)

    def audit(self, role):
        rows=[];targets=[]
        for row in self.records(role):
            m=self.metadata(row)
            rows.append(dict(subject=row['subject'],file=row['relative_path'],windows=len(m['label']),
                             valid_signal=int(m['valid_signal'].sum()),prediction_ready=int(m['prediction_ready'].sum()),
                             **{f'ready_label_{k}':int((m['prediction_ready']&(m['label']==k)).sum()) for k in [-1,0,1,2]}))
        for e in self.events:
            if not e['eligible_target'] or e['split']!='pool' or e['subject'] not in self.roles[role]:continue
            a=e['onset_s']-self.cfg['sph_s']-self.cfg['sop_s'];b=e['onset_s']-self.cfg['sph_s']
            n=0
            for row in self.records(role):
                if row['case']!=e['case']:continue
                m=self.metadata(row)
                # Decision times, independent of supervised labels.
                n+=int((m['prediction_ready']&(m['decision_s']>=a)&(m['decision_s']<=b)).sum())
            targets.append(dict(event_id=e['event_id'],subject=e['subject'],retain_in_evaluation=True,
                                available_forecast_decisions=n,has_forecast_opportunity=n>0))
        return pd.DataFrame(rows),pd.DataFrame(targets)


import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

class CoreMVModel(nn.Module):
    """Explicit smoke-test foundation, not the complete physics-informed model."""
    def __init__(self, channels=18, latent=32, rank=8):
        super().__init__()
        self.raw_conv=nn.Sequential(nn.Conv1d(channels,channels,7,padding=3,groups=channels),nn.GELU(),
                                    nn.Conv1d(channels,64,1),nn.GELU())
        self.raw_project=nn.Linear(128,latent)
        self.spectral=nn.Sequential(nn.Flatten(1),nn.Linear(channels*5,64),nn.LayerNorm(64),nn.GELU(),nn.Linear(64,latent))
        self.fusion=nn.Sequential(nn.Linear(latent*2,latent),nn.LayerNorm(latent),nn.GELU())
        self.classifier=nn.Linear(latent,3)
        self.U=nn.Parameter(torch.randn(latent,rank)*0.01)
        self.V=nn.Parameter(torch.randn(latent,rank)*0.01)

    def encode(self,raw,spec):
        h=self.raw_conv(raw)
        zr=self.raw_project(torch.cat([h.mean(-1),h.amax(-1)],dim=-1))
        zs=self.spectral(spec)
        return self.fusion(torch.cat([zr,zs],dim=-1))

    def propagate(self,z):
        # Row-vector convention for K=I+UV^T: z_next = z K^T.
        return z+(z@self.V)@self.U.T

    def forward(self,raw,spec):
        z=self.encode(raw,spec)
        return self.classifier(z),z

    def operator_matrix(self):
        return torch.eye(self.U.shape[0],device=self.U.device)+self.U@self.V.T


def tensor_batch(samples,device,paired=False):
    def stack(group,key):
        return torch.as_tensor(np.stack([s[group][key] for s in samples]),dtype=torch.float32,device=device)
    out={'raw':stack('inputs','raw_t'),'spec':stack('inputs','spec_t'),
         'labels':torch.tensor([s['target']['label'] for s in samples],dtype=torch.long,device=device)}
    if paired:
        out.update(next_raw=stack('auxiliary','raw_tdt'),next_spec=stack('auxiliary','spec_tdt'))
    return out


def core_loss(model,batch,dynamics_weight=0.5):
    logits,z=model(batch['raw'],batch['spec'])
    ce=F.cross_entropy(logits,batch['labels'])
    with torch.no_grad():
        target=model.encode(batch['next_raw'],batch['next_spec'])
    dynamics=F.mse_loss(model.propagate(z),target)
    return ce+dynamics_weight*dynamics,{'classification':ce,'dynamics':dynamics},z


def check_gradients(model):
    report={}
    for name,p in model.named_parameters():
        if p.grad is None or not torch.isfinite(p.grad).all():
            raise RuntimeError(f'Missing/nonfinite gradient: {name}')
        report[name]=float(p.grad.norm().detach().cpu())
    if not any(x>0 for x in report.values()): raise RuntimeError('All gradients are zero')
    return report


@torch.no_grad()
def causal_probabilities(model, sample, device, top_k=8, encode_chunk=16):
    if not sample['metadata']['prediction_ready']:
        return None
    inputs=sample['inputs']
    # Manuscript convention: last 74 preceding windows + current = 75.
    # Loader requires 75 preceding valid windows, so its availability rule is
    # conservatively one window stricter. Preserve it; never silently relax it.
    H=len(inputs['raw_past'])
    if H<top_k or H<1: raise ValueError('Insufficient history for Top-K score')
    raw=np.concatenate([inputs['raw_past'][1:],inputs['raw_t'][None]],axis=0)
    spec=np.concatenate([inputs['spec_past'][1:],inputs['spec_t'][None]],axis=0)
    probabilities=[]
    was_training=model.training;model.eval()
    try:
        for start in range(0,len(raw),encode_chunk):
            logits,_=model(torch.as_tensor(raw[start:start+encode_chunk],device=device),
                           torch.as_tensor(spec[start:start+encode_chunk],device=device))
            probabilities.append(logits.softmax(-1).cpu())
    finally:
        model.train(was_training)
    p=torch.cat(probabilities).numpy()
    return dict(p_interictal=float(p[-1,0]),p_preictal=float(p[-1,1]),p_ictal=float(p[-1,2]),
                top8_preictal=float(np.sort(p[:,1])[-top_k:].mean()))


def smoke_sample_indices(data,per_class=2,seed=20261028):
    # Deliberately tiny class-stratified engineering test, not a training sampler.
    candidates=data.index('train','pairs');rng=np.random.default_rng(seed)
    picked=[]
    for label in [0,1,2]:
        eligible=[x for x in candidates if data.metadata(x[0])['label'][x[1]]==label]
        if not eligible: raise RuntimeError(f'No class {label} dynamics pairs in this training split')
        chosen=rng.choice(len(eligible),size=min(per_class,len(eligible)),replace=False)
        picked.extend(eligible[int(i)] for i in chosen)
    return picked


## 3 · Integration and loss coupling
Integrate the rate model over the actual adjacent-window interval $\Delta t=4$ seconds using 16 substeps. Exponential Euler freezes the nonlinear activation at each substep and solves the leak exactly:

$$s_{k+1}=e^{-h/\tau}s_k+(1-e^{-h/\tau})f(s_k),\quad h=\Delta t/16.$$

This convex combination preserves [0,1] without clipping; **boundedness does not prove convergence or dynamical stability**. Compare 16, 32 and 256 substeps on the sampled EEG batch. The smoke-test tolerance is maximum absolute state difference <0.005 between 16 and 256 substeps. A failure requires inspecting/refining integration; do not silently loosen the tolerance.

Let $K=I+UV^\top$ and $\Phi(s_t)$ denote the rate-model transition. Define:

$$L_{dyn}=\mathrm{MSE}(Kz_t,\operatorname{stopgrad}(z_{t+1})),$$
$$L_{coupling}=\mathrm{MSE}(\sigma(Kz_t),\Phi(\sigma(z_t))),$$
$$L_{empirical}=\mathrm{MSE}(\Phi(\sigma(z_t)),\operatorname{stopgrad}(\sigma(z_{t+1}))),$$
$$L_{physics}=(L_{coupling}+L_{empirical})/2,$$
$$L=L_{CE}+0.5L_{dyn}+0.2L_{physics}.$$

The coupling loss directly reaches both K and the rate model. The empirical term also asks the rate transition to fit the next observed encoded state. The next-window branch is detached; the current encoders receive gradients. The classifier continues to use current z only. **Both dynamics branches are auxiliary training regularizers, not additional inference inputs.**

Weights 0.5 and 0.2 are smoke-test settings, not selected scientific results. Representation collapse remains possible; the notebook reports latent variation/effective rank and sigmoid saturation. No stability penalty, decoder or reconstruction objective is silently added. The low-rank K remains unconstrained in spectral radius.

In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F

class WCInspired(nn.Module):
    """16 uncoupled formal E/I pairs sharing 8 parameters; effective seconds.
    Reduced sigmoid rate model, not the full original Wilson-Cowan equations.
    """
    def __init__(self):
        super().__init__()
        self.weight_logits=nn.Parameter(torch.logit(torch.tensor([1.5,1.,1.,0.5])/4))
        self.tau_logits=nn.Parameter(torch.logit((torch.tensor([10.,16.])-4)/36))
        self.drive_logits=nn.Parameter(torch.atanh(torch.tensor([-.5,-.5])/2))

    def physical_parameters(self):
        return 4*torch.sigmoid(self.weight_logits),4+36*torch.sigmoid(self.tau_logits),2*torch.tanh(self.drive_logits)

    def activation(self,state):
        E,I=state.chunk(2,dim=-1)
        w,tau,b=self.physical_parameters()
        return torch.cat([torch.sigmoid(w[0]*E-w[1]*I+b[0]),
                          torch.sigmoid(w[2]*E-w[3]*I+b[1])],dim=-1)

    def rhs(self,state):
        _,tau,_=self.physical_parameters()
        rate=torch.cat([tau[0].expand(state.shape[-1]//2),tau[1].expand(state.shape[-1]//2)])
        return (self.activation(state)-state)/rate

    def forward(self,state,dt_s=4.,substeps=16):
        if state.shape[-1]!=32:raise ValueError('Expected 16 E + 16 I coordinates')
        if substeps<1 or int(substeps)!=substeps or not math.isfinite(dt_s) or dt_s<=0:
            raise ValueError('Invalid integration settings')
        if not torch.isfinite(state).all() or torch.any(state<0) or torch.any(state>1):
            raise ValueError('Initial state must be finite and in [0,1]')
        _,tau,_=self.physical_parameters()
        rate=torch.cat([tau[0].expand(16),tau[1].expand(16)])
        # Exponential Euler: exact leak, activation frozen per substep.
        # Convex combination preserves [0,1] without clipping. This is NOT
        # an assertion of global dynamical stability or numerical convergence.
        decay=torch.exp(-(dt_s/int(substeps))/rate)
        y=state
        for _ in range(int(substeps)):
            y=decay*y+(1-decay)*self.activation(y)
        return y

class PhysicsMVModel(CoreMVModel):
    def __init__(self,channels=18,latent=32,rank=8):
        if latent!=32:raise ValueError('Physics v1 requires latent=32')
        super().__init__(channels,latent,rank)
        self.wc=WCInspired()


def physics_loss(model,batch,dt_s=4.,substeps=16,dynamics_weight=.5,physics_weight=.2):
    logits,z=model(batch['raw'],batch['spec'])
    with torch.no_grad():
        next_z=model.encode(batch['next_raw'],batch['next_spec'])
        next_state=torch.sigmoid(next_z)
    kz=model.propagate(z)
    wc_state=model.wc(torch.sigmoid(z),dt_s=dt_s,substeps=substeps)
    classification=F.cross_entropy(logits,batch['labels'])
    dynamics=F.mse_loss(kz,next_z)
    coupling=F.mse_loss(torch.sigmoid(kz),wc_state)
    empirical=F.mse_loss(wc_state,next_state)
    physics=(coupling+empirical)/2
    total=classification+dynamics_weight*dynamics+physics_weight*physics
    return total,dict(classification=classification,dynamics=dynamics,
                      physics_coupling=coupling,physics_empirical=empirical,physics=physics),z

@torch.no_grad()
def physics_diagnostics(model,batch,dt_s=4.):
    z=model.encode(batch['raw'],batch['spec'])
    state=torch.sigmoid(z)
    y16=model.wc(state,dt_s,16);y32=model.wc(state,dt_s,32);y256=model.wc(state,dt_s,256)
    centered=z.double()-z.double().mean(0)
    eigen=torch.linalg.svdvals(centered).square()
    if eigen.sum()>1e-20:
        p=eigen/eigen.sum();p=p[p>0];rank=float(torch.exp(-(p*p.log()).sum()).cpu())
    else:rank=0.
    K=model.operator_matrix().detach().double().cpu()
    w,tau,b=model.wc.physical_parameters()
    return dict(state_min=float(y16.min().cpu()),state_max=float(y16.max().cpu()),
                max_difference_16_vs_32=float((y16-y32).abs().max().cpu()),
                max_difference_16_vs_256=float((y16-y256).abs().max().cpu()),
                max_difference_32_vs_256=float((y32-y256).abs().max().cpu()),
                latent_mean_std=float(z.std(0,unbiased=False).mean().cpu()),
                latent_effective_rank=rank,effective_rank_upper_bound=min(len(z)-1,z.shape[-1]),
                state_saturated_fraction=float(((state<.01)|(state>.99)).float().mean().cpu()),
                koopman_spectral_radius=float(torch.linalg.eigvals(K).abs().max()),
                koopman_largest_singular_value=float(torch.linalg.svdvals(K).max()),
                wc_weights=w.cpu().tolist(),wc_tau_effective_seconds=tau.cpu().tolist(),wc_drives=b.cpu().tolist())


## 4 · Synthetic numerical and gradient tests
These run independently of your EEG. They check the known analytic uncoupled solution, integration refinement, signs, gradients, direct coupling to K, and recovery of the core loss when the physics weight is zero.

In [ ]:
import torch
import numpy as np

def run_physics_tests():
    results=[]
    with torch.random.fork_rng():
        torch.manual_seed(11)
        wc=WCInspired().double()
        x=torch.rand(10,32,dtype=torch.double);x[0]=0;x[1]=1
        out=wc(x)
        assert torch.isfinite(out).all() and out.min()>=0 and out.max()<=1
        w,t,b=wc.physical_parameters();assert (w>0).all() and (w<4).all() and (t>4).all() and (t<40).all()
        results.append('bounded states and constrained parameters')
        # With all interactions zero, the exact solution is known.
        with torch.no_grad():wc.weight_logits.fill_(-1000)
        y=wc(x);_,tau,drive=wc.physical_parameters()
        decay=torch.cat([torch.exp(-4/tau[0]).expand(16),torch.exp(-4/tau[1]).expand(16)])
        steady=torch.cat([torch.sigmoid(drive[0]).expand(16),torch.sigmoid(drive[1]).expand(16)])
        expected=decay*x+(1-decay)*steady
        torch.testing.assert_close(y,expected,atol=1e-12,rtol=1e-12)
        results.append('analytic uncoupled solution')
        wc=WCInspired().double()
        y16=wc(x,4,16);y32=wc(x,4,32);ref=wc(x,4,2048)
        assert (y32-ref).abs().max()<(y16-ref).abs().max()
        assert (y16-ref).abs().max()<.005
        results.append('integration refinement against 2048 substeps')
        # Check signs of cross-population influence in the vector field.
        x0=torch.full((1,32),.4,dtype=torch.double,requires_grad=True)
        J=torch.autograd.functional.jacobian(wc.rhs,x0)[0,:,0,:]
        assert J[0,16]<0 and J[16,0]>0
        results.append('inhibitory-to-E negative, excitatory-to-I positive')
        assert torch.autograd.gradcheck(lambda a:wc(a,4,4),(x0,),atol=1e-5,rtol=1e-3)
        results.append('integration input gradient finite-difference check')
        m=PhysicsMVModel(channels=2)
        batch=dict(raw=torch.randn(6,2,1024),spec=torch.randn(6,2,5),labels=torch.tensor([0,0,1,1,2,2]),
                   next_raw=torch.randn(6,2,1024,requires_grad=True),next_spec=torch.randn(6,2,5,requires_grad=True))
        loss,parts,z=physics_loss(m,batch)
        loss.backward();check_gradients(m)
        assert batch['next_raw'].grad is None and batch['next_spec'].grad is None
        assert all(p.grad.norm()>0 for p in m.wc.parameters())
        results.append('full loss gradients and detached future observation')
        # Physics alone must reach K and current encoder, not just fit the WC cell.
        m.zero_grad(set_to_none=True);_,parts,_=physics_loss(m,batch)
        parts['physics'].backward()
        assert m.U.grad.norm()>0 and m.V.grad.norm()>0
        assert m.raw_project.weight.grad.norm()>0
        assert all(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.norm()>0 for p in m.wc.parameters())
        results.append('physics-only gradient reaches operator, encoders and rate model')
        m.zero_grad(set_to_none=True)
        a,_,_=physics_loss(m,batch,physics_weight=0)
        b,_,_=core_loss(m,batch)
        torch.testing.assert_close(a,b)
        results.append('zero physics weight recovers the core loss')
        before=m(batch['raw'],batch['spec'])[0].detach().clone()
        with torch.no_grad():m.wc.drive_logits.add_(2)
        after=m(batch['raw'],batch['spec'])[0].detach()
        torch.testing.assert_close(before,after)
        results.append('physics parameters cannot directly alter inference logits')
        d=physics_diagnostics(m,batch)
        assert d['latent_effective_rank']<=5.0001 and d['max_difference_16_vs_256']<.005
        results.append('rank diagnostics respect small-batch limit')
    return results


synthetic_tests=run_physics_tests()
for name in synthetic_tests: print('PASS:',name)
print('All',len(synthetic_tests),'test groups passed.')

## 5 · Initialize a fresh model and select training pairs
Uses the same development fold as your earlier smoke test. Up to two examples per class exercise the code; this tiny stratified selection is not the eventual sampling strategy. Metadata scanning over Drive can take several minutes.

In [ ]:
import random
SEED=20261028
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cpu':torch.set_num_threads(min(4,torch.get_num_threads()))
RUN=Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
OUTER='outer_chb01';SCOPE='inner_01'
data=FoldData(RUN,OUTER,SCOPE)
if SCOPE=='outer_refit':raise RuntimeError('Use an inner development split for this check.')
if data.cfg['window_s']!=4:raise RuntimeError('This documented prototype assumes four-second adjacent windows.')
model=PhysicsMVModel(channels=len(data.cfg['channels'])).to(DEVICE)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
chosen=smoke_sample_indices(data,per_class=2,seed=SEED)
samples=[data.read(row,i,mode='pairs') for row,i in chosen]
batch=tensor_batch(samples,DEVICE,paired=True)
print('Device:',DEVICE,'training labels:',batch['labels'].tolist())
print('Training subjects:',data.roles['train'])
print('Total parameters:',sum(p.numel() for p in model.parameters()))
print('Additional rate-model parameters:',sum(p.numel() for p in model.wc.parameters()))
assert sum(p.numel() for p in model.wc.parameters())==8
print('Model initialized fresh; no smoke checkpoint was loaded.')

## 6 · Three training steps with physics enabled
Checks finite, nonzero gradients in each branch and the rate-model parameters. The physics-only gradient path is separately checked on the actual batch before optimization. Decreasing loss is not required over just three steps; finite updates are the engineering criterion.

In [ ]:
model.train()
optimizer.zero_grad(set_to_none=True)
_,parts,_=physics_loss(model,batch)
parts['physics'].backward()
physics_only_norms={k:float(p.grad.norm().detach().cpu()) if p.grad is not None else 0.
                    for k,p in model.named_parameters()}
for key in ['U','V','raw_project.weight','wc.weight_logits','wc.tau_logits','wc.drive_logits']:
    if not np.isfinite(physics_only_norms[key]) or physics_only_norms[key]<=0:
        raise RuntimeError('No finite physics-only gradient at '+key)
print('Physics-only gradient paths verified.')
initial={k:p.detach().clone() for k,p in model.named_parameters()}
training_log=[]
for step in range(3):
    optimizer.zero_grad(set_to_none=True)
    loss,parts,z=physics_loss(model,batch)
    if not torch.isfinite(loss):raise RuntimeError('Nonfinite total loss')
    loss.backward();norms=check_gradients(model)
    for branch in ['raw_conv','raw_project','spectral','fusion','classifier','wc']:
        if not any(v>0 for k,v in norms.items() if k.startswith(branch+'.')):
            raise RuntimeError('No gradient in '+branch)
    grad=torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
    optimizer.step()
    if not all(torch.isfinite(p).all() for p in model.parameters()):raise RuntimeError('Nonfinite parameters')
    training_log.append(dict(step=step+1,total=float(loss.detach().cpu()),
                             **{k:float(v.detach().cpu()) for k,v in parts.items()},
                             preclip_gradient_norm=float(grad.detach().cpu())))
changed=[k for k,p in model.named_parameters() if not torch.equal(initial[k],p.detach())]
if not any(k.startswith('wc.') for k in changed):raise RuntimeError('Rate-model parameters did not update')
display(pd.DataFrame(training_log))
print('Parameter tensors updated:',len(changed),'/',len(initial))

## 7 · Integration and representation diagnostics
The effective rank of six centered samples cannot exceed five, even with a 32-dimensional latent. This diagnostic cannot establish whole-dataset non-collapse. K may have spectral radius >1: report it, do not label this model stable. The state bounds apply only to the sigmoid-rate transition.

In [ ]:
model.eval()
diagnostics=physics_diagnostics(model,batch)
print(json.dumps(diagnostics,indent=2))
assert diagnostics['state_min']>=0 and diagnostics['state_max']<=1
if diagnostics['max_difference_16_vs_256']>=.005:
    raise RuntimeError('Integration refinement exceeds the stated 0.005 tolerance. Review before further training.')
if not np.isfinite(diagnostics['latent_mean_std']) or diagnostics['latent_mean_std']<=1e-8:
    raise RuntimeError('Latent variation is effectively zero on this batch.')
print('Integration check passed on this training batch. Full-data convergence and collapse remain to be assessed.')

## 8 · Save diagnostics and explicit modeling choices
Creates a fresh folder. Checkpoint is debug-only; it is not a trained forecasting model. Send the loss table, diagnostics, and final summary. Original preprocessing, folds and event-scoring protocol remain unchanged.

In [ ]:
from datetime import datetime,timezone
OUTPUT=RUN/'physics_smoke_v1'/OUTER/SCOPE/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT.mkdir(parents=True,exist_ok=False)
summary=dict(stage='wc_inspired_physics_smoke_v1',status='physics_smoke_passed_full_training_not_run',
             outer=OUTER,scope=SCOPE,seed=SEED,device=str(DEVICE),torch_version=str(torch.__version__),
             training_steps=3,training_samples=len(chosen),parameter_tensors_updated=len(changed),
             total_parameters=sum(p.numel() for p in model.parameters()),rate_model_parameters=8,
             dt_seconds=4,integration='exponential Euler',substeps=16,refinement_substeps=256,tolerance=.005,
             state_mapping='fixed sigmoid; first 16 formal E, last 16 formal I',
             physiology_identified=False,rate_model='reduced shared-pair sigmoid-rate prior; no refractory terms',
             weight_bounds=[0,4],tau_bounds_effective_seconds=[4,40],drive_bounds=[-2,2],
             loss='CE + 0.5*latent_dynamics + 0.2*(state_coupling + empirical_state_transition)/2',
             next_state_target_detached=True,operator_spectral_stability_guaranteed=False,
             inference='current fused latent classifier; physics acts through training only',
             synthetic_tests=synthetic_tests,diagnostics=diagnostics,
             scaler_sha256=hashlib.sha256(data.scaler_path.read_bytes()).hexdigest(),
             folds_sha256=hashlib.sha256((RUN/'locked_folds.json').read_bytes()).hexdigest(),
             training_sample_ids=[dict(subject=r['subject'],file=r['relative_path'],index=i) for r,i in chosen],
             real_validation_metrics_computed=False,outer_test_accessed=False)
(OUTPUT/'physics_smoke_summary.json').write_text(json.dumps(summary,indent=2))
(OUTPUT/'physics_only_gradient_norms.json').write_text(json.dumps(physics_only_norms,indent=2))
pd.DataFrame(training_log).to_csv(OUTPUT/'physics_training_losses.csv',index=False)
torch.save(dict(model_state_dict=model.state_dict(),model_config=dict(channels=len(data.cfg['channels']),latent=32,rank=8),
                smoke_only=True,seed=SEED),OUTPUT/'debug_only_physics_checkpoint.pt')
print(json.dumps(summary,indent=2))
print('Saved:',OUTPUT)

## Handoff to the pilot
After this passes, the next experiment is a matched development-only pilot: classifier alone, classifier plus Koopman dynamics, and classifier plus dynamics plus this prior. Keep the encoder/classifier initialization, data split, training budget and scoring policy comparable. Reinitialize models; do not continue these six-example checkpoints. Setting physics weight to zero recovers the core objective exactly.

The prior's sign constraints do not establish biological identifiability. A benefit must be demonstrated by ablation; ideally include a parameter-matched generic dynamics control. Large-dataset latent rank, state saturation, integration refinement over training trajectories and operator diagnostics must be monitored. The prototype's weights and time-scale bounds remain experimental choices and can only be changed using development data.

No threshold has been selected, no held-out performance has been computed, and no inference of physiological excitation/inhibition is justified by passing these tests.